# 365 Probabilidades - Dia #162
## Qual a probabilidade de pegar friagem e ficar resfriado?

**Tipo:** Ditado testado com dado  
**Data de publicação:** 2026-11-22  
**Ferramenta:** Python  
**Decisão analisada:** Vale levar o casaco e calçar o chinelo, como a gente ouviu a vida inteira?  
**Hashtag:** #365Probabilidades #Dia162

---

### 📖 A História

Leva um casaco. Não sai de cabelo molhado. Calça um chinelo, que esse chão está gelado.

Quase todo mundo cresceu ouvindo alguma dessas frases, e muita gente hoje repete. A resposta de quem estudou um pouco de biologia também é conhecida. Resfriado é vírus, e vírus não vem do frio.

As duas coisas são ditas com a mesma certeza.

Fui ver o que os estudos mais recentes mostram.

---

### 📚 O Conceito

A pergunta da friagem pode ser estudada de três jeitos, e cada um responde a uma parte dela.

- **Na população.** Conta-se quanta gente adoece nos dias frios e nos dias amenos. O número de pessoas é enorme, mas o que se mede é a temperatura da cidade, e não o frio que cada pessoa passou.
- **No experimento.** Pessoas são sorteadas, e parte delas passa frio de propósito. O sorteio permite falar em causa, mas a amostra é menor e o frio é de laboratório.
- **No laboratório.** Mede-se o que o frio faz com as defesas do nariz. Mostra um mecanismo possível, e não prova que alguém adoece por isso.

O dia junta os três.

---

### 🧮 O Modelo

Cinco estudos, de 2005 a 2024, uma base pública brasileira e seis camadas.

1. **Quando a cidade esfria.** O risco relativo de Seul traduzido em atendimentos a cada 100 e na parcela ligada ao frio (200 mil sorteios, semente 42).
2. **Quando a temperatura cai.** A razão de chances por grau, da Finlândia, projetada para quedas de 1 a 10 graus, com intervalo.
3. **Friagem de propósito.** O ensaio do banho frio, com a chance de a água fria aumentar os dias doente e a diferença na parcela de pessoas que adoeceram.
4. **Dentro do nariz.** Os números do estudo de laboratório, sem conta nova.
5. **A friagem do ditado.** O experimento dos pés na água fria, com intervalo de Jeffreys e o valor de p refeito com um caso de diferença.
6. **E no Brasil.** As internações por síndrome respiratória aguda grave em São Paulo, Curitiba e Porto Alegre, de 2013 a 2018, cruzadas com a temperatura de cada semana. Primeiro a comparação entre semanas frias e quentes. Depois uma regressão de quase-Poisson que compara cada semana com as outras semanas do mesmo mês e do mesmo ano, com a temperatura da própria semana e das três anteriores. É o mesmo tipo de comparação do estudo de Seul. As três cidades são combinadas pelo inverso da variância.

**Fontes:**
- Lee, H. & Yoon, H. Y. (2024). Impact of ambient temperature on respiratory disease: a case-crossover study in Seoul. *Respiratory Research*, 25, 73. **1.616.644 atendimentos** de emergência por doença respiratória em Seul, de 2008 a 2017. Lido no artigo completo.
- Ikäheimo, T. M., Jaakkola, K., Jokelainen, J., Saukkoriipi, A., Roivainen, M., Juvonen, R., Vainio, O. & Jaakkola, J. J. K. (2016). A decrease in temperature and humidity precedes human rhinovirus infections in a cold climate. *Viruses*, 8(9), 244. **892 recrutas** finlandeses, 147 casos de rinovírus confirmados por exame. Lido no artigo completo.
- Buijze, G. A., Sierevelt, I. N., van der Heijden, B. C. J. M., Dijkgraaf, M. G. & Frings-Dresen, M. H. W. (2016). The effect of cold showering on health and work: a randomized controlled trial. *PLOS ONE*, 11(9), e0161749. **3.018 adultos** sorteados, 2.426 analisados aos 90 dias. Lido no artigo completo.
- Huang, D., Taha, M. S., Nocera, A. L., Workman, A. D., Amiji, M. M. & Bleier, B. S. (2023). Cold exposure impairs extracellular vesicle swarm-mediated nasal antiviral immunity. *Journal of Allergy and Clinical Immunology*, 151(2), 509-525. Estudo de laboratório com células e tecido do nariz, e medição em voluntários. Lido no artigo completo.
- Johnson, C. & Eccles, R. (2005). Acute cooling of the feet and the onset of common cold symptoms. *Family Practice*, 22(6), 608-613. **180 voluntários** sorteados. **Lido no resumo. O artigo completo não foi lido.**
- Ministério da Saúde, OpenDataSUS. Banco de dados de síndrome respiratória aguda grave (SRAG), 2013 a 2018, arquivos INFLUD13 a INFLUD18. **201.799 registros** no país, dos quais **30.698 internações** de moradores de São Paulo, Curitiba e Porto Alegre. Temperatura semanal da NASA POWER (parâmetros T2M e T2M_MIN). Dados baixados em 12/11/2026 e agregados por semana no arquivo `dia-162-srag-temperatura-semanal.csv`, que acompanha o notebook.

**Nota metodológica sobre o fator ×0,80.** Não se aplica. As fontes são registros de atendimento, exame de laboratório e experimentos com sorteio. Os dias doente do ensaio do banho frio e o resfriado do estudo de 2005 são relatados pelos participantes, e esse limite está tratado no texto e nas limitações.

**Corda declarada:** #113 (abraço e resfriado, Cohen e colegas 2015). Lá os voluntários receberam o vírus. Aqui a pergunta é sobre o frio.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
SEMENTE, N_SIM = 42, 200_000
rng = np.random.default_rng(SEMENTE)

def jeffreys(k, n, nivel=0.95):
    """Intervalo de Jeffreys para uma proporção."""
    a = (1 - nivel) / 2
    return stats.beta.ppf([a, 1 - a], k + 0.5, n - k + 0.5)

def sorteia(k, n):
    return rng.beta(k + 0.5, n - k + 0.5, N_SIM)

print("✅ Bibliotecas carregadas")

In [ ]:
# --- DADOS DA LITERATURA ---

# 1) Lee, H. & Yoon, H. Y. (2024). Impact of ambient temperature on respiratory disease: a case-crossover
#    study in Seoul. Respiratory Research, 25, 73. Lido no artigo completo (Tabelas 1 e 2).
seul_atendimentos = 1_616_644          # atendimentos de emergência por doença respiratória, 2008 a 2017
seul_media_dia_vias_superiores = 253.5 # média diária de atendimentos por infecção aguda das vias aéreas superiores
seul_frio = (2.00, 1.68, 2.39)         # risco relativo a -9,0 °C contra 24,3 °C (temperatura de menor risco), IC 95%
seul_calor = (1.23, 1.08, 1.40)        # risco relativo a 29,4 °C contra 24,3 °C
seul_todas_frio = (2.68, 2.26, 3.17)   # todas as doenças respiratórias, a -9,0 °C

# 2) Ikäheimo, T. M. et al. (2016). A decrease in temperature and humidity precedes human rhinovirus
#    infections in a cold climate. Viruses, 8(9), 244. Lido no artigo completo (Tabela 2).
fin_recrutas, fin_casos = 892, 147     # recrutas finlandeses; casos de rinovírus confirmados por exame
fin_or_por_grau = (1.08, 1.01, 1.17)   # razão de chances por 1 °C de queda nos três dias anteriores (ajustada)

# 3) Buijze, G. A. et al. (2016). The effect of cold showering on health and work: a randomized
#    controlled trial. PLOS ONE, 11(9), e0161749. Lido no artigo completo (Tabelas 2 e 4).
banho_sorteados = 3018
banho_analisados = {'30 segundos': 673, '60 segundos': 611, '90 segundos': 595, 'Controle': 547}   # aos 90 dias
banho_pct_doente = {'30 segundos': 65.0, '60 segundos': 63.3, '90 segundos': 64.5, 'Controle': 69.3}  # % com algum dia doente
banho_pct_falta = {'30 segundos': 29.4, '60 segundos': 34.0, '90 segundos': 33.1, 'Controle': 34.8}   # % com alguma falta
banho_irr_dias_doente = (0.89, 0.77, 1.01)   # razão de taxas, água fria contra controle, p = 0,073
banho_irr_faltas = (0.71, 0.56, 0.89)        # p = 0,003

# 4) Huang, D. et al. (2023). Cold exposure impairs extracellular vesicle swarm-mediated nasal antiviral
#    immunity. Journal of Allergy and Clinical Immunology, 151(2), 509-525. Lido no artigo completo.
nariz_ambiente = (23.3, 4.4)                 # °C, antes e depois
nariz_queda = {'parte da frente': 6.4, 'parte do meio': 4.7}   # queda máxima da temperatura dentro do nariz, °C
nariz_reducao = {'vesículas liberadas': 41.9, 'material antiviral\nem cada vesícula': 23.8,
                 'ligação ao vírus\n(receptor 1)': 24.4, 'ligação ao vírus\n(receptor 2)': 77.2}   # redução em %

# 5) Johnson, C. & Eccles, R. (2005). Acute cooling of the feet and the onset of common cold symptoms.
#    Family Practice, 22(6), 608-613. Lido no resumo.
n_por_grupo = 90
resfriado_pes_frios, resfriado_controle = 13, 5      # relataram resfriado em até cinco dias; p publicado = 0,047

aplica_fator_080 = False   # registros, exame de laboratório e experimentos com sorteio

fmt = lambda x, c=1: f"{x:.{c}f}".replace('.', ',')
print("=" * 72)
print("  DADOS DA LITERATURA - FRIAGEM E RESFRIADO")
print("=" * 72)
print(f"\n  Seul (Lee & Yoon, 2024), {seul_atendimentos:,} atendimentos de emergência".replace(',', '.'))
print(f"  → Infecção das vias aéreas superiores a -9 °C: risco {fmt(seul_frio[0], 2)} vezes [{fmt(seul_frio[1], 2)}; {fmt(seul_frio[2], 2)}]")
print(f"  → No calor de 29,4 °C: {fmt(seul_calor[0], 2)} vezes [{fmt(seul_calor[1], 2)}; {fmt(seul_calor[2], 2)}]")
print(f"\n  Finlândia (Ikäheimo et al., 2016), {fin_recrutas} recrutas, {fin_casos} casos de rinovírus")
print(f"  → Cada 1 °C de queda: chance {fmt(fin_or_por_grau[0], 2)} vezes [{fmt(fin_or_por_grau[1], 2)}; {fmt(fin_or_por_grau[2], 2)}]")
print(f"\n  Banho frio (Buijze et al., 2016), {banho_sorteados} sorteados, {sum(banho_analisados.values())} analisados aos 90 dias".replace(',', '.'))
for g in banho_analisados:
    print(f"  → {g:<12} n = {banho_analisados[g]}  ·  {fmt(banho_pct_doente[g])}% com algum dia doente  ·  {fmt(banho_pct_falta[g])}% com alguma falta")
print(f"  → Dias doente: razão {fmt(banho_irr_dias_doente[0], 2)} [{fmt(banho_irr_dias_doente[1], 2)}; {fmt(banho_irr_dias_doente[2], 2)}]  ·  faltas: {fmt(banho_irr_faltas[0], 2)} [{fmt(banho_irr_faltas[1], 2)}; {fmt(banho_irr_faltas[2], 2)}]")
print("\n  Nariz (Huang et al., 2023)")
print(f"  → Ambiente de {fmt(nariz_ambiente[0])} °C para {fmt(nariz_ambiente[1])} °C: o nariz esfria até {fmt(nariz_queda['parte da frente'])} °C na frente e {fmt(nariz_queda['parte do meio'])} °C no meio")
print(f"  → Vesículas de defesa liberadas: {fmt(nariz_reducao['vesículas liberadas'])}% a menos")
print(f"\n  Pés na água fria (Johnson & Eccles, 2005), {2*n_por_grupo} sorteados")
print(f"  → {resfriado_pes_frios} de {n_por_grupo} contra {resfriado_controle} de {n_por_grupo} relataram resfriado")
print(f"\n  Fator ×0,80 aplicado: {'sim' if aplica_fator_080 else 'não'}")

# 6) Camada brasileira, com dados públicos.
#    Internações por síndrome respiratória aguda grave (SRAG): Ministério da Saúde, OpenDataSUS,
#    banco de SRAG de 2013 a 2018 (arquivos INFLUD13 a INFLUD18), por município de residência
#    e semana dos primeiros sintomas. Temperatura: NASA POWER (reanálise por satélite), média semanal.
#    O arquivo dia-162-srag-temperatura-semanal.csv precisa estar na mesma pasta do notebook.
br_tudo = np.genfromtxt('dia-162-srag-temperatura-semanal.csv', delimiter=',', names=True, dtype=None, encoding='utf-8')
br = br_tudo[br_tudo['semana_inicio'] != '2014-12-28']     # semana sem nenhum registro nas três cidades (virada dos arquivos)
cidades = {'São Paulo': 'sao_paulo', 'Curitiba': 'curitiba', 'Porto Alegre': 'porto_alegre'}
br_registros_pais = 201_799                      # registros nos seis arquivos, país inteiro
br_total = {c: int(br['srag_' + k].sum()) for c, k in cidades.items()}

print("\n  Brasil (OpenDataSUS e NASA POWER), 2013 a 2018")
print(f"  → {len(br)} semanas  ·  " + "  ·  ".join(f"{c}: {v:,} internações".replace(',', '.') for c, v in br_total.items()))
print(f"  → Total nas três cidades: {sum(br_total.values()):,}".replace(',', '.'))
print("=" * 72)

In [ ]:
# --- O MODELO ---
# Cinco contas próprias sobre os números publicados. Semente 42, 200 mil sorteios.

def sorteia_razao(est, lo, hi):
    """Sorteia uma razão (risco, chances ou taxas) a partir do IC 95% publicado, na escala do logaritmo."""
    ep = (np.log(hi) - np.log(lo)) / (2 * 1.96)
    return np.exp(rng.normal(np.log(est), ep, N_SIM))

# Camada 1 · Quando a cidade esfria (Seul)
sim_seul = sorteia_razao(*seul_frio)
a_mais_em_100 = (seul_frio[0] - 1) * 100                       # atendimentos a mais para cada 100 de um dia ameno
fatia_do_frio = 1 - 1 / sim_seul                               # parte dos atendimentos do dia frio ligada ao frio
ic_fatia = np.percentile(fatia_do_frio, [2.5, 50, 97.5]) * 100

# Camada 2 · Quando a temperatura cai (Finlândia)
sim_fin = sorteia_razao(*fin_or_por_grau)
p_fin = (sim_fin > 1).mean()
quedas = np.arange(1, 11)
curva_fin = np.array([np.percentile(sim_fin ** q, [2.5, 50, 97.5]) for q in quedas])
queda5 = curva_fin[4]; queda10 = curva_fin[9]

# Camada 3 · Friagem de propósito (banho frio)
sim_irr_doente = sorteia_razao(*banho_irr_dias_doente)
p_banho_piora = (sim_irr_doente > 1).mean()                    # chance de o banho frio aumentar os dias doente
p_banho_melhora = 1 - p_banho_piora
grupos_frios = ['30 segundos', '60 segundos', '90 segundos']
n_frio = sum(banho_analisados[g] for g in grupos_frios)
k_frio = sum(int(round(banho_pct_doente[g] / 100 * banho_analisados[g])) for g in grupos_frios)   # reconstruído
n_ctrl = banho_analisados['Controle']; k_ctrl = int(round(banho_pct_doente['Controle'] / 100 * n_ctrl))
dif_banho = (sorteia(k_frio, n_frio) - sorteia(k_ctrl, n_ctrl)) * 100
ic_dif_banho = np.percentile(dif_banho, [2.5, 50, 97.5])
p_menos_doentes = (dif_banho < 0).mean()

# Camada 4 · Dentro do nariz: só os números publicados, sem conta nova

# Camada 5 · A friagem do ditado (pés na água fria)
sim_frio = sorteia(resfriado_pes_frios, n_por_grupo); sim_ctrl = sorteia(resfriado_controle, n_por_grupo)
p_frio, p_ctrl = resfriado_pes_frios / n_por_grupo, resfriado_controle / n_por_grupo
ic_pes, ic_c = jeffreys(resfriado_pes_frios, n_por_grupo), jeffreys(resfriado_controle, n_por_grupo)
dif_pes = (sim_frio - sim_ctrl) * 100
ic_dif_pes = np.percentile(dif_pes, [2.5, 50, 97.5])
p_friagem_aumenta = (dif_pes > 0).mean()
pessoas_por_resfriado = 1 / (p_frio - p_ctrl)
def p_fisher(a, b, n=n_por_grupo): return stats.fisher_exact([[a, n - a], [b, n - b]])[1]
def p_qui(a, b, n=n_por_grupo): return stats.chi2_contingency([[a, n - a], [b, n - b]], correction=False)[1]
cenarios = {'um caso a menos\nnos pés frios': (resfriado_pes_frios - 1, resfriado_controle),
            'como publicado': (resfriado_pes_frios, resfriado_controle),
            'um caso a mais\nno controle': (resfriado_pes_frios, resfriado_controle + 1)}
p_cen = {k: (p_qui(*v), p_fisher(*v)) for k, v in cenarios.items()}

print("=" * 72)
print("  MODELO - RESULTADO")
print("=" * 72)
print("\n  CAMADA 1 · Quando a cidade esfria (Seul)")
print(f"  → Para cada 100 atendimentos de um dia ameno, {fmt(seul_frio[0]*100, 0)} num dia de -9 °C [{fmt(seul_frio[1]*100, 0)}; {fmt(seul_frio[2]*100, 0)}]")
print(f"  → Num dia de -9 °C, {fmt(ic_fatia[1], 0)}% dos atendimentos estão ligados ao frio [{fmt(ic_fatia[0], 0)}; {fmt(ic_fatia[2], 0)}]")
print("\n  CAMADA 2 · Quando a temperatura cai (Finlândia)")
print(f"  → P(a queda de temperatura aumenta a chance de rinovírus) = {fmt(p_fin*100, 0)}%")
print(f"  → Queda de 5 °C: chance {fmt(queda5[1], 2)} vezes [{fmt(queda5[0], 2)}; {fmt(queda5[2], 2)}]")
print(f"  → Queda de 10 °C: chance {fmt(queda10[1], 2)} vezes [{fmt(queda10[0], 2)}; {fmt(queda10[2], 2)}]")
print("\n  CAMADA 3 · Friagem de propósito (banho frio)")
print(f"  → Dias doente: razão {fmt(banho_irr_dias_doente[0], 2)}  ·  P(o banho frio aumenta os dias doente) = {fmt(p_banho_piora*100, 0)}%  ·  P(diminui) = {fmt(p_banho_melhora*100, 0)}%")
print(f"  → Com algum dia doente em 90 dias: {k_frio} de {n_frio} = {fmt(k_frio/n_frio*100)}% (água fria) contra {k_ctrl} de {n_ctrl} = {fmt(k_ctrl/n_ctrl*100)}% (controle)")
print(f"  → Diferença {fmt(ic_dif_banho[1])} pp [{fmt(ic_dif_banho[0])}; {fmt(ic_dif_banho[2])}]  ·  P(menos gente doente com água fria) = {fmt(p_menos_doentes*100, 0)}%")
print(f"  → Faltas ao trabalho por doença: {fmt((1-banho_irr_faltas[0])*100, 0)}% a menos [{fmt((1-banho_irr_faltas[2])*100, 0)}; {fmt((1-banho_irr_faltas[1])*100, 0)}]")
print("\n  CAMADA 5 · A friagem do ditado (pés na água fria)")
print(f"  → {fmt(p_frio*100)}% [{fmt(ic_pes[0]*100)}; {fmt(ic_pes[1]*100)}] contra {fmt(p_ctrl*100)}% [{fmt(ic_c[0]*100)}; {fmt(ic_c[1]*100)}]")
print(f"  → Diferença {fmt(ic_dif_pes[1])} pp [{fmt(ic_dif_pes[0])}; {fmt(ic_dif_pes[2])}]  ·  P(friagem aumenta o relato de resfriado) = {fmt(p_friagem_aumenta*100, 0)}%")
print(f"  → Um resfriado a mais a cada {fmt(pessoas_por_resfriado, 0)} pessoas  ·  {fmt((1-p_frio)*100, 0)}% de quem pegou friagem não relatou resfriado")
for k, (pq, pf) in p_cen.items():
    a, b = cenarios[k]
    print(f"  → {k.replace(chr(10), ' '):<30} {a} contra {b}  ·  p = {fmt(pq, 3)} (qui-quadrado)  ·  p = {fmt(pf, 3)} (Fisher)")

# Camada 6 · E no Brasil (três capitais, 2013 a 2018)
from datetime import date, timedelta
quartas = [date.fromisoformat(s) + timedelta(days=3) for s in br_tudo['semana_inicio']]
br_ano = np.array([d.year for d in quartas]); br_mes = np.array([d.month for d in quartas])
br_sem = np.array([min((d.timetuple().tm_yday - 1) // 7, 51) for d in quartas])
valida = br_tudo['semana_inicio'] != '2014-12-28'

def quase_poisson(y, X):
    # Regressão de Poisson com dispersão estimada (quase-Poisson), por mínimos quadrados reponderados.
    b = np.linalg.lstsq(X, np.log(y + 0.5), rcond=None)[0]
    for _ in range(100):
        mu = np.exp(X @ b); z = X @ b + (y - mu) / mu
        b_novo = np.linalg.solve(X.T @ (X * mu[:, None]), X.T @ (mu * z))
        if np.abs(b_novo - b).max() < 1e-10: b = b_novo; break
        b = b_novo
    mu = np.exp(X @ b); phi = ((y - mu) ** 2 / mu).sum() / (len(y) - X.shape[1])
    return b, np.linalg.inv(X.T @ (X * mu[:, None])) * phi, phi

QUEDA, DEFASAGENS = 5, 4      # 5 °C a menos · a própria semana e as três seguintes

def efeito_do_frio(chave, coluna='temp_media_', sem_ano=None):
    # Compara cada semana com as outras semanas do mesmo mês e do mesmo ano (estratos de ano e mês),
    # como no estudo de Seul. A temperatura entra com a própria semana e as três anteriores.
    y = br_tudo['srag_' + chave].astype(float); t = br_tudo[coluna + chave].astype(float)
    idx = np.arange(DEFASAGENS - 1, len(y))
    usa = valida[idx] & ((br_ano[idx] != sem_ano) if sem_ano else True)
    T = np.column_stack([t[idx - d] for d in range(DEFASAGENS)])[usa]
    estrato = np.unique(br_ano[idx][usa] * 100 + br_mes[idx][usa], return_inverse=True)[1]
    X = np.column_stack([T, np.eye(estrato.max() + 1)[estrato]])
    b, V, phi = quase_poisson(y[idx][usa], X)
    b, V = b[:DEFASAGENS], V[:DEFASAGENS, :DEFASAGENS]
    log_ac = -QUEDA * b.sum(); ep = QUEDA * np.sqrt(V.sum())
    return dict(por_semana=np.exp(-QUEDA * b), log=log_ac, ep=ep, phi=phi,
                razao=(np.exp(log_ac), np.exp(log_ac - 1.96 * ep), np.exp(log_ac + 1.96 * ep)))

def junta(res):
    # Média das cidades ponderada pelo inverso da variância (efeito fixo).
    w = np.array([1 / r['ep'] ** 2 for r in res]); m = (w * np.array([r['log'] for r in res])).sum() / w.sum(); ep = 1 / np.sqrt(w.sum())
    return (np.exp(m), np.exp(m - 1.96 * ep), np.exp(m + 1.96 * ep)), stats.norm.cdf(m / ep)

br_res = {}
for c, k in cidades.items():
    y = br['srag_' + k].astype(float); t = br['temp_media_' + k].astype(float)
    q1, q3 = np.quantile(t, [0.25, 0.75])
    a_, s_ = br_ano[valida], br_sem[valida]
    pico = [int(s_[a_ == a][np.argmax(y[a_ == a])]) for a in range(2013, 2019)]
    mais_fria = [int(s_[a_ == a][np.argmin(t[a_ == a])]) for a in range(2013, 2019)]
    br_res[c] = dict(frias=y[t <= q1].mean(), quentes=y[t >= q3].mean(), antes=sum(p < f for p, f in zip(pico, mais_fria)),
                     base=efeito_do_frio(k), minima=efeito_do_frio(k, 'temp_min_'), sem2016=efeito_do_frio(k, sem_ano=2016))
    br_res[c]['razao'] = br_res[c]['frias'] / br_res[c]['quentes']
br_junto, p_br = junta([r['base'] for r in br_res.values()])
br_junto_min, _ = junta([r['minima'] for r in br_res.values()])
br_junto_sem2016, _ = junta([r['sem2016'] for r in br_res.values()])
picos_antes = sum(r['antes'] for r in br_res.values())
tres = lambda v: f"{fmt(v[0], 2)} [{fmt(v[1], 2)}; {fmt(v[2], 2)}]"

print("\n  CAMADA 6 · E no Brasil (internações por SRAG, 2013 a 2018)")
for c, r in br_res.items():
    print(f"  {c}")
    print(f"  → As 25% de semanas mais frias: {fmt(r['frias'])} internações por semana  ·  as 25% mais quentes: {fmt(r['quentes'])}  ·  razão {fmt(r['razao'], 1)}")
    print(f"  → Semana 5 °C mais fria que as outras do mesmo mês: internações × {tres(r['base']['razao'])}, efeito acumulado em quatro semanas")
    print(f"     por semana: " + "  ".join(fmt(v, 2) for v in r['base']['por_semana']) + f"  ·  dispersão {fmt(r['base']['phi'])}")
    print(f"     com a temperatura mínima: {tres(r['minima']['razao'])}  ·  sem 2016: {tres(r['sem2016']['razao'])}")
print(f"  As três cidades juntas")
print(f"  → Internações × {tres(br_junto)}  ·  P(o frio aumenta as internações) {'acima de 99,9' if p_br > 0.999 else '= ' + fmt(p_br*100, 1)}%")
print(f"  → Com a temperatura mínima: {tres(br_junto_min)}  ·  sem 2016: {tres(br_junto_sem2016)}")
print(f"  → Pico das internações antes da semana mais fria do ano em {picos_antes} de 18 pares de cidade e ano")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---

# ── GRÁFICO 1 · Seul ──
fig1, ax1 = plt.subplots(figsize=(12, 8))
itens = [('Dia ameno\n(24,3 °C)', (1, 1, 1), CINZA), ('Dia quente\n(29,4 °C)', seul_calor, DOURADO), ('Dia muito frio\n(-9 °C)', seul_frio, VERMELHO)]
for i, (rot, (e, lo, hi), cor) in enumerate(itens):
    ax1.bar(i, e * 100, 0.55, color=cor, alpha=0.85)
    if hi > lo:
        ax1.errorbar(i, e * 100, yerr=[[(e - lo) * 100], [(hi - e) * 100]], color='#222', capsize=8, lw=1.8)
    ax1.text(i, hi * 100 + 6, f'{e*100:.0f}', ha='center', fontsize=18, fontweight='bold')
ax1.set_xticks(range(3)); ax1.set_xticklabels([it[0] for it in itens], fontsize=13)
ax1.set_ylim(0, 280)
ax1.set_ylabel('Atendimentos por infecção das vias aéreas superiores\n(dia ameno = 100)', fontsize=12)
ax1.set_title('Num dia muito frio, Seul atende o dobro de infecções das vias aéreas\n'
              '1,6 milhão de atendimentos de emergência por doença respiratória, 2008 a 2017', fontsize=14, pad=15)
plt.figtext(0.5, 0.01, 'Fonte: Lee & Yoon, 2024, Respiratory Research. Efeito acumulado em até 21 dias; barras com IC 95% | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-162-grafico-01-seul.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 · Finlândia ──
fig2, ax2 = plt.subplots(figsize=(12, 8))
ax2.fill_between(quedas, curva_fin[:, 0], curva_fin[:, 2], color=VERDE, alpha=0.2, label='intervalo de 95%')
ax2.plot(quedas, curva_fin[:, 1], color=VERDE, lw=3, marker='o', label='estimativa')
ax2.axhline(1, color='#222', ls='--', lw=1.3)
ax2.text(10.05, 1.02, 'sem efeito', fontsize=11, color='#333', ha='right')
for q in (5, 10):
    ax2.annotate(f'{curva_fin[q-1, 1]:.1f} vezes'.replace('.', ','), (q, curva_fin[q-1, 1]), textcoords='offset points', xytext=(-10, 14), fontsize=14, fontweight='bold', ha='right')
ax2.set_xticks(quedas)
ax2.set_xlabel('Queda de temperatura nos três dias anteriores (°C)', fontsize=12)
ax2.set_ylabel('Chance de infecção por rinovírus (vezes)', fontsize=12)
ax2.set_title('Quanto mais a temperatura cai, maior a chance de rinovírus confirmado por exame\n'
              f'{fin_casos} casos entre {fin_recrutas} recrutas finlandeses  ·  P(a queda aumenta a chance) = {p_fin*100:.0f}%', fontsize=14, pad=15)
ax2.legend(fontsize=12, loc='upper left')
plt.figtext(0.5, 0.01, 'Modelo próprio sobre Ikäheimo et al., 2016 (razão de chances de 1,08 por grau, IC de 1,01 a 1,17) | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-162-grafico-02-finlandia.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 · Banho frio ──
fig3, ax3 = plt.subplots(figsize=(12, 8))
nomes = list(banho_analisados.keys())
for i, g in enumerate(nomes):
    n = banho_analisados[g]; k = int(round(banho_pct_doente[g] / 100 * n)); ic = jeffreys(k, n) * 100; v = banho_pct_doente[g]
    ax3.bar(i, v, 0.55, color=CINZA if g == 'Controle' else VERDE, alpha=0.85)
    ax3.errorbar(i, v, yerr=[[v - ic[0]], [ic[1] - v]], color='#222', capsize=8, lw=1.8)
    ax3.text(i, ic[1] + 1.2, f'{v:.0f}%', ha='center', fontsize=16, fontweight='bold')
ax3.set_xticks(range(4)); ax3.set_xticklabels([f'{g} de\nágua fria\n(n = {banho_analisados[g]})' if g != 'Controle' else f'Banho normal\n(n = {banho_analisados[g]})' for g in nomes], fontsize=12)
ax3.set_ylim(0, 90)
ax3.set_ylabel('Tiveram algum dia doente em 90 dias (%)', fontsize=12)
ax3.set_title('Quem terminou o banho com água fria todo dia não adoeceu mais\n'
              f'{banho_sorteados:,} adultos sorteados  ·  P(o banho frio aumenta os dias doente) = {p_banho_piora*100:.0f}%'.replace(',', '.'), fontsize=14, pad=15)
plt.figtext(0.5, 0.01, 'Fonte: Buijze et al., 2016, PLOS ONE. Contagens reconstruídas dos percentuais; barras com intervalo de Jeffreys de 95% | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-162-grafico-03-banho-frio.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ── GRÁFICO 4 · Dentro do nariz ──
fig4, ax4 = plt.subplots(figsize=(12, 8))
rot4 = list(nariz_reducao.keys()); val4 = list(nariz_reducao.values())
b = ax4.barh(rot4, val4, color=VERMELHO, alpha=0.85, height=0.55)
for r, v in zip(b, val4):
    ax4.text(v + 1.5, r.get_y() + r.get_height() / 2, f'{v:.0f}% a menos', va='center', fontsize=14, fontweight='bold')
ax4.invert_yaxis(); ax4.set_xlim(0, 100)
ax4.tick_params(axis='y', labelsize=12)
ax4.set_xlabel('Redução quando o tecido do nariz esfria 5 °C (%)', fontsize=12)
ax4.set_title('O que o frio faz com uma defesa do nariz contra vírus\n'
              f'Com o ar a {nariz_ambiente[1]:.1f} °C, o interior do nariz esfria cerca de 5 °C'.replace('.', ','), fontsize=14, pad=15)
plt.figtext(0.5, 0.01, 'Fonte: Huang et al., 2023, Journal of Allergy and Clinical Immunology. Estudo de laboratório | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-162-grafico-04-dentro-do-nariz.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

# ── GRÁFICO 5 · A friagem do ditado ──
fig5, (axa, axb) = plt.subplots(1, 2, figsize=(14, 7))
val = [p_ctrl * 100, p_frio * 100]; icg = [ic_c * 100, ic_pes * 100]; cont = [resfriado_controle, resfriado_pes_frios]
for i, cor in enumerate([CINZA, VERMELHO]):
    axa.bar(i, val[i], 0.5, color=cor, alpha=0.85)
    axa.errorbar(i, val[i], yerr=[[val[i] - icg[i][0]], [icg[i][1] - val[i]]], color='#222', capsize=8, lw=1.8)
    axa.text(i, icg[i][1] + 0.8, f'{cont[i]} de {n_por_grupo}', ha='center', fontsize=15, fontweight='bold')
axa.set_xticks([0, 1]); axa.set_xticklabels(['Sem friagem', 'Pés na água fria'], fontsize=12)
axa.set_ylim(0, 30); axa.set_xlim(-0.6, 1.6)
axa.set_ylabel('Relataram resfriado em até cinco dias (%)', fontsize=12)
axa.set_title('O experimento', fontsize=13, pad=10)
nomes5 = list(cenarios.keys()); pq = [p_cen[k][0] for k in nomes5]; pf = [p_cen[k][1] for k in nomes5]
x = np.arange(3); larg = 0.36
axb.bar(x - larg / 2, pq, larg, color=DOURADO, alpha=0.85, label='qui-quadrado')
axb.bar(x + larg / 2, pf, larg, color=VERDE, alpha=0.85, label='teste exato de Fisher')
for i in range(3):
    axb.text(x[i] - larg / 2, pq[i] + 0.004, f'{pq[i]:.3f}'.replace('.', ','), ha='center', fontsize=11, fontweight='bold', bbox=dict(facecolor='white', edgecolor='none', pad=1))
    axb.text(x[i] + larg / 2, pf[i] + 0.004, f'{pf[i]:.3f}'.replace('.', ','), ha='center', fontsize=11, fontweight='bold')
axb.axhline(0.05, color=VERMELHO, ls='--', lw=1.6)
axb.set_xticks(x); axb.set_xticklabels([f'{n}\n({cenarios[n][0]} contra {cenarios[n][1]})' for n in nomes5], fontsize=10)
axb.set_ylim(0, max(pf) * 1.3); axb.set_ylabel('Valor de p', fontsize=12)
axb.set_title('O mesmo resultado com um caso de diferença', fontsize=13, pad=10)
axb.legend(fontsize=10, loc='upper left')
fig5.suptitle(f'A friagem do ditado: um efeito modesto, que depende de um único caso', fontsize=15)
plt.figtext(0.5, 0.01, 'Fonte: Johnson & Eccles, 2005 (180 voluntários sorteados). Valores de p recalculados aqui | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('dia-162-grafico-05-a-friagem-do-ditado.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 5 salvo!")

# GRÁFICO 6 · A estação fria no Brasil
fig6, ax6 = plt.subplots(figsize=(12, 8))
nomes = list(br_res); xs = np.arange(len(nomes)); larg = 0.36
b1 = ax6.bar(xs - larg/2, [br_res[c]['quentes'] for c in nomes], larg, color=DOURADO, label='As 25% de semanas mais quentes')
b2 = ax6.bar(xs + larg/2, [br_res[c]['frias'] for c in nomes], larg, color=VERDE, label='As 25% de semanas mais frias')
for b in list(b1) + list(b2):
    ax6.text(b.get_x() + b.get_width()/2, b.get_height() + 1, fmt(b.get_height(), 0), ha='center', fontsize=14, fontweight='bold')
ax6.set_xticks(xs); ax6.set_xticklabels([f"{c}\nrazão de {fmt(br_res[c]['razao'], 1)}" for c in nomes], fontsize=13)
ax6.set_ylabel('Internações por SRAG por semana (média)'); ax6.set_ylim(0, 95); ax6.legend(frameon=False, loc='upper right')
ax6.set_title('Nas semanas frias há mais internações por doença respiratória grave\nTrês capitais, 2013 a 2018', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, 'Fonte: Ministério da Saúde (OpenDataSUS, SRAG 2013 a 2018) e NASA POWER. Conta própria | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-162-grafico-06-brasil-estacao.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 6 salvo!")

# GRÁFICO 7 · A semana mais fria que as vizinhas
fig7, ax7 = plt.subplots(figsize=(12, 8))
rot = nomes + ['As três juntas']; vals = [br_res[c]['base']['razao'] for c in nomes] + [br_junto]
cores7 = [DOURADO, DOURADO, DOURADO, VERDE]
for i, (v, cor) in enumerate(zip(vals, cores7)):
    ax7.errorbar(i, v[0], yerr=[[v[0] - v[1]], [v[2] - v[0]]], fmt='o', color=cor, markersize=13, capsize=7, linewidth=2.5)
    ax7.text(i + 0.09, v[0], f"{fmt(v[0], 2)}\n[{fmt(v[1], 2)}; {fmt(v[2], 2)}]", va='center', fontsize=12, fontweight='bold' if i == 3 else 'normal')
ax7.axhline(1, color=CINZA, linestyle='--', linewidth=1.5)
ax7.text(-0.4, 1.012, 'sem diferença', fontsize=10, color=CINZA)
ax7.set_xticks(range(4)); ax7.set_xticklabels(rot, fontsize=13); ax7.set_xlim(-0.5, 3.7); ax7.set_ylim(0.8, 1.95)
ax7.set_ylabel('Internações em relação às outras semanas do mesmo mês (1 = igual)')
ax7.set_title('Quando a temperatura cai 5 °C, as internações sobem nas semanas seguintes\nEfeito acumulado em quatro semanas, com intervalo de 95%', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, 'Fonte: Ministério da Saúde (OpenDataSUS, SRAG 2013 a 2018) e NASA POWER. Conta própria | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-162-grafico-07-brasil-semana-fria.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 7 salvo!")

### 💡 O Insight

**Num dia de 9 graus negativos, Seul atendeu o dobro de infecções das vias aéreas de um dia ameno. E num ensaio com 3.018 pessoas, quem tomou água fria de propósito todo dia não adoeceu mais.**

Os dois resultados são recentes, grandes e parecem opostos. Eles medem coisas diferentes.

Quando a temperatura cai, mais gente adoece. Em Seul foram 200 atendimentos para cada 100 de um dia ameno, com intervalo de 168 a 239. Entre recrutas finlandeses, cada grau de queda aumentou em 8% a chance de um rinovírus confirmado por exame, e uma queda de 5 graus corresponde a uma chance 1,5 vez maior.

O laboratório mostra um caminho possível. Com o ar a 4 °C, o interior do nariz esfria cerca de 5 graus, e as células do nariz liberam 42% menos das vesículas que ajudam a barrar vírus.

Já o frio curto e de propósito não fez mal. Quem terminou o banho com 30 a 90 segundos de água fria, por um mês, teve 64% de chance de algum dia doente em três meses, contra 69% de quem tomou banho normal. A chance de o banho frio aumentar os dias doente é de 5%.

A friagem do ditado, a dos pés gelados, foi testada uma vez, em 2005. Relataram resfriado 13 de 90 pessoas, contra 5 de 90 sem friagem. É um resfriado a mais a cada 11 pessoas, e o resultado depende de um único caso para ficar de pé.

No Brasil, as internações por doença respiratória grave são mais numerosas nas semanas frias. Separei, em cada cidade, as 25% de semanas mais frias e as 25% mais quentes. Em São Paulo foram 78 internações por semana nas mais frias, contra 52 nas mais quentes. Em Curitiba foram 32 contra 10, e em Porto Alegre 38 contra 10. A queda de temperatura dentro do mesmo mês também aparece. Quando a temperatura fica 5 graus abaixo das outras semanas do mês, as internações sobem 29% ao longo de quatro semanas, com intervalo de 14% a 45%, juntando as três cidades. É o mesmo sentido de Seul e da Finlândia, com um efeito menor.

A resposta que os dados permitem é esta. O frio que dura dias e esfria o ar que se respira está ligado a mais resfriado. Um minuto de água fria ou um chão gelado, pelo que se mediu até hoje, pesa pouco.

A pergunta que fica:

*Quantas das regras que você ouviu em casa foram testadas alguma vez, e por quantas pessoas?*

---

### ⚠️ Limitações do Modelo

- As cinco fontes medem coisas diferentes (atendimento de emergência, vírus confirmado, dias doente relatados, células em laboratório, sintoma relatado). Os números não devem ser somados nem comparados ponto a ponto.
- **Seul.** O estudo mede a temperatura da cidade, não o frio que cada pessoa passou. O grupo analisado é o das infecções agudas das vias aéreas superiores (CID J00 a J06), que inclui o resfriado e outras infecções. Atendimento de emergência não é o mesmo que doença, e no frio as pessoas também ficam mais tempo em lugares fechados. O risco é acumulado em até 21 dias e compara −9 °C com 24,3 °C, que é a temperatura de menor risco naquela cidade. É associação.
- **Seul.** O texto do artigo traz o risco das vias aéreas superiores como 1,68, e a Tabela 2 traz 2,00 com intervalo de 1,68 a 2,39. Usamos a tabela, que é coerente com o intervalo.
- **Finlândia.** São 147 casos em recrutas jovens, num clima em que a temperatura média do período foi de −9,9 °C. A razão de chances é ajustada e tem intervalo que encosta no 1. O resumo e a tabela do artigo trazem valores ligeiramente diferentes (1,07 e 1,08), e usamos a tabela. A projeção para quedas de 5 e 10 graus é conta nossa e supõe que o efeito de cada grau se multiplica.
- **Banho frio.** Os participantes sabiam em que grupo estavam, e os dias doente e as faltas foram relatados por eles. Eram adultos muito saudáveis (96% com saúde boa ou excelente), e 20% não responderam aos 90 dias. Para os dias doente, os autores não encontraram efeito significativo (razão de 0,89, p = 0,073). A redução de 29% é nas faltas ao trabalho.
- **Banho frio.** As contagens de quem teve algum dia doente foram reconstruídas dos percentuais publicados. A comparação da água fria com o controle junta os três grupos e não usa o modelo dos autores. O ensaio testa o frio breve e repetido, que a pessoa escolheu, e não a friagem inesperada do ditado.
- **Nariz.** É estudo de laboratório, com células e tecido retirado em cirurgia. Mostra que uma defesa diminui no frio, e não que a pessoa adoece por isso. O número de voluntários da medição de temperatura está no material suplementar, que não foi lido.
- **Pés na água fria.** Lido no resumo. O desfecho é o relato do voluntário, sem exame de vírus e sem como esconder o grupo. O valor de p publicado (0,047) corresponde ao qui-quadrado. O teste exato de Fisher, refeito aqui, dá 0,080. Não encontramos réplica.
- A probabilidade de 98% no estudo de 2005 e a de 5% no banho frio são contas nossas sobre os números publicados. Elas respondem se o efeito tem um sentido, e não se ele é grande.
- **Brasil.** A medida é a internação por síndrome respiratória aguda grave, que está longe do resfriado comum. Ela inclui gripe e outros vírus, depende de a pessoa ser internada e de o caso ser notificado, e os critérios de notificação mudaram ao longo dos anos. A data usada é a dos primeiros sintomas.
- **Brasil.** A temperatura é a da NASA POWER, uma estimativa por satélite e modelo para o ponto central de cada cidade, e não a de uma estação meteorológica. É a temperatura da cidade, e não o frio que cada pessoa passou.
- **Brasil.** A comparação entre semanas frias e quentes mistura o frio com tudo o que muda na estação (circulação de vírus, aulas, ambientes fechados, umidade). É associação.
- **Brasil.** O efeito de 29% é a soma dos efeitos da própria semana e das três seguintes, numa regressão com dados semanais. Os estudos da área usam dados diários e modelos mais flexíveis. Sozinha, São Paulo tem intervalo que inclui o 1 (de 0,93 a 1,81) e dispersão alta, por causa da epidemia de gripe de 2016, que começou em março com tempo quente. Curitiba encosta no 1. O resultado conjunto depende mais de Porto Alegre.
- **Brasil.** O resultado muda com as escolhas. Com a temperatura mínima no lugar da média, o efeito conjunto é de 21% (7% a 35%). Sem o ano de 2016, é de 40% (23% a 59%). Uma primeira conta mais simples, que comparava cada semana com a média daquela semana do ano em seis anos, não encontrou efeito. Ela foi abandonada porque seis anos são poucos para estimar o normal de cada semana. Em 12 dos 18 pares de cidade e ano, o pico das internações veio antes da semana mais fria, o que mostra que a temperatura não é o único fator da estação. A semana de 28/12/2014 não tem registro em nenhuma das três cidades e foi retirada.
- Nenhum estudo testou cabelo molhado, vento ou chão gelado, que são as situações do ditado.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*